# Phase 1 — Data & Observability Audit

Audit event-aligned StatsBomb 360 observations across the 34 Leverkusen Bundesliga matches. This notebook presents derived diagnostics; reusable audit logic lives in `leverkusen.data.observability`.

360 frames are not continuous tracking. `teammate=True/False` retains raw provider semantics. No attacking direction, player identity, defensive regime or final visibility threshold is inferred.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import Markdown, display

from leverkusen.data.observability import (
    audit_season,
    largest_actor_discrepancies,
    summarize_actor_distances,
)

ROOT = Path.cwd()
if not (ROOT / "config/project.yaml").is_file():
    ROOT = ROOT.parent
project = yaml.safe_load((ROOT / "config/project.yaml").read_text(encoding="utf-8"))
diagnostics = ROOT / "outputs/diagnostics"

Run `python scripts/audit_data.py` to refresh the three derived CSVs. If they exist, this notebook reads them without network requests; their retrieval dates are shown below. If any is missing, it runs the same reusable audit in memory. It never writes raw JSON or regenerates the CSVs automatically.

In [ ]:
paths = [diagnostics / f"phase1_{name}.csv" for name in ("match_summary", "frame_summary", "attrition")]
if all(path.is_file() for path in paths):
    matches, frames, attrition = [pd.read_csv(path) for path in paths]
else:
    matches, frames, attrition = audit_season(
        team_name=project["team_name"],
        pitch_length=project["coordinate_length"],
        pitch_width=project["coordinate_width"],
    )
season = attrition.loc[attrition["scope"].eq("season")].iloc[0]
display(matches[["retrieved_at_utc", "source_base_url"]].agg(["min", "max"]))

## 1. Match coverage

Inventory and 360 load status are separate from frame-level visibility. A missing or failed 360 response leaves the match frame count unknown, not zero.

In [ ]:
assert len(matches) == matches["match_id"].nunique() == 34
display(matches[["match_id", "date", "opponent", "venue", "match_week", "total_events", "frames_load_status", "total_frames"]].sort_values("date"))
display(matches["frames_load_status"].value_counts())
display(matches.loc[~matches["frames_loaded"], ["match_id", "frames_load_error"]])

## 2. Event ↔ 360 coverage and integrity

Event counts use original event rows. Frame counts use original frame rows. Duplicate identifiers do not expand a many-to-many join; ambiguous and orphan frame rows remain visible.

In [ ]:
integrity_columns = [
    "total_events", "unique_event_ids", "matched_events", "events_without_frames",
    "total_frames", "unique_frame_event_uuids", "frames_without_matching_events",
    "duplicate_event_ids", "duplicate_frame_uuids", "frames_with_ambiguous_event",
    "events_missing_id", "frames_missing_uuid",
]
display(matches.set_index("match_id")[integrity_columns])
display(attrition.loc[attrition["scope"].eq("event_type"), ["group", "all_events", "events_with_360_frame", "all_frames"]])
# Explicitly expose matches with no usable UUID link rather than hiding them in totals.
display(matches.loc[matches["matched_events"].eq(0), [
    "match_id", "date", "opponent", "total_events", "total_frames",
    "frames_load_status", "frames_without_matching_events",
]])

## 3. Visible-player distributions

Counts are raw teammate flags, keepers and actors. Missing frames have no rows; missing freeze-frame containers have unknown counts. An observed empty list has zero players. Unknown flags are not reassigned.

In [ ]:
player_columns = ["visible_players", "teammate_true", "teammate_false", "keepers", "actors", "teammate_unknown", "valid_player_locations"]
display(frames[player_columns].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))
frames[["visible_players", "teammate_true", "teammate_false"]].plot.hist(bins=23, alpha=0.5, figsize=(10, 4))
plt.xlabel("Observed player count per frame")
plt.ylabel("Frames")
plt.title("Visible players and raw teammate flags")
plt.show()

## 4. Visible-area coverage

Valid polygons retain their original area. Pitch coverage uses the polygon intersection with the 120 × 80 pitch divided by 9,600. Invalid polygons are not repaired. Coordinate count counts scalar x/y entries; vertex count counts pairs, including an explicit closing point if supplied.

In [ ]:
display(frames["visible_area_reason"].value_counts(dropna=False))
display(frames[["visible_area_polygon_area", "visible_area_on_pitch_area", "visible_area_fraction"]].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))
frames["visible_area_fraction"].dropna().plot.hist(bins=30, figsize=(9, 4))
plt.xlabel("Fraction of pitch in valid visible-area polygon")
plt.ylabel("Frames")
plt.title("Pitch coverage — no eligibility cutoff")
plt.show()

## 5. Actor-coordinate consistency

Distance is measured only for a uniquely linked event with a finite location and exactly one located actor. These are coordinate units, not automatically metres. Largest distances are review priorities, not threshold failures; no coordinates are flipped or corrected.

In [ ]:
display(pd.Series(summarize_actor_distances(frames), name="Coordinate units"))
display(frames["actor_consistency_reason"].value_counts(dropna=False))
display(largest_actor_discrepancies(frames)[[
    "match_id", "frame_ordinal", "event_id", "event_index", "event_type", "event_team",
    "actors", "actor_distance", "visible_players", "visible_area_fraction",
]])
frames["actor_distance"].dropna().plot.hist(bins=50, figsize=(9, 4))
plt.yscale("log")
plt.xlabel("Actor/event distance in original coordinates")
plt.ylabel("Frames (log scale)")
plt.title("Actor consistency, including the full discrepancy tail")
plt.show()

## 6. Sample attrition

The first table contains independent event/frame counts, not a single funnel. The second is cumulative within frame rows: unique event and frame identifiers → nonempty frame → valid area → measurable actor and valid player locations. Passing these mechanical checks does not establish final research eligibility. Grouped summaries are alternative partitions and must not be added across scopes.

In [ ]:
independent = ["all_events", "events_with_360_frame", "events_360_load_unavailable", "all_frames", "frames_nonempty_freeze_frame", "frames_valid_visible_area", "frames_actor_measurable"]
cumulative = ["all_frames", "frames_unique_event_and_frame_id", "frames_unique_nonempty", "frames_unique_nonempty_valid_area", "frames_joint_checks"]
display(season[independent].rename("Independent counts"))
display(season[cumulative].rename("Cumulative mechanical checks"))
for scope in ("match_id", "event_type", "event_team_group"):
    display(Markdown(f"### By {scope}"))
    display(attrition.loc[attrition["scope"].eq(scope), ["group", *independent, "frames_joint_checks", "actor_distance_max"]])

## 7. Observations and remaining questions

The following observations are regenerated from the displayed audit tables. Full 360 availability by match does not imply complete event coverage or complete player visibility.

In [ ]:
display(Markdown(
    f"- Confirmed **{matches['match_id'].nunique():,} unique matches**; "
    f"360 loaded for **{int(matches['frames_loaded'].sum()):,}**.\n"
    f"- **{int(season['events_with_360_frame']):,} / {int(season['all_events']):,} events** "
    f"have a frame ({season['events_with_360_frame'] / season['all_events']:.2%}).\n"
    f"- **{int(season['frames_valid_visible_area']):,} / {int(season['all_frames']):,} frames** "
    f"have valid visible-area polygons.\n"
    f"- Actor distance is measurable for **{int(season['frames_actor_measurable']):,} frames**; "
    f"median **{season['actor_distance_median']:.4f}**, maximum **{season['actor_distance_max']:.4f}** coordinate units.\n"
    f"- **{int(season['frames_joint_checks']):,} frames** pass the joint mechanical checks; "
    "this is not a calibrated usable-sample claim."
))

Before spatial geometry, resolve or explicitly exclude any match with zero event/frame UUID overlap, inspect the largest actor discrepancies, and review event types with systematically absent frames. Full-season availability is not enough to establish full-season event-linked usability.

The current live run has zero UUID overlap in matches **3895158, 3895266 and 3895309** (10,213 orphan frames). Four frames in match **3895139** have two actor flags. Actor distance has a near-zero median but a 95th percentile of **36.416486** coordinate units. These findings are recorded in `report/technical_appendix.md`; refresh this commentary if the upstream release changes.

Resolve event/frame coordinate conventions and raw teammate semantics before general spatial analysis. Player visibility, goalkeeper inclusion, missing/multiple-actor treatment and any distance tolerance remain research decisions. Null calibration files are unchanged. The dataset is not yet ready for unrestricted event-linked spatial geometry. This notebook stops at Phase 1.